# Sci-Plex scRNA-seq Drug Perturbation Signatures

This notebook contains the script used to process the Sci-Plex Drug Perturbation Signatures dataset for Harmonizome.

The Sci-Plex 3 files were downloaded from [GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4150378) on 11/4/25.


In [ ]:
import pandas as pd
import datetime
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import re
import requests
import scipy.spatial.distance as dist
import seaborn as sns
import sys
import json
import scanpy as sc
from scipy.sparse import csr_matrix
from tqdm import tqdm
from maayanlab_bioinformatics.normalization import quantile_normalize
from maayanlab_bioinformatics.normalization.filter import filter_by_expr
from maayanlab_bioinformatics.dge.characteristic_direction import characteristic_direction
from urllib.request import urlretrieve
import pyreadr

# UMAP
from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()

from IPython.display import display, HTML, Markdown
sys.setrecursionlimit(100000)

## Load and Pre-Process Data

### sciPlex3

In [ ]:
umi_counts = pd.read_csv('sciPlex3/GSM4150378_sciPlex3_A549_MCF7_K562_screen_UMI.count.matrix.gz', header=None, sep='\t', compression='gzip')
umi_counts.columns = ['Gene', 'Cell', 'UMI']
umi_counts

In [ ]:
# dose in nM
sample_metadata = pd.read_csv('sciPlex3/GSM4150378_sciPlex3_A549_MCF7_K562_hashTable_metadata.txt.gz', sep='\t', compression='gzip')
sample_metadata = sample_metadata[~sample_metadata['vehicle']].set_index('drug_dose')[['name', 'dose']].drop_duplicates()
sample_metadata

In [ ]:
gene_annotations = pd.read_csv('sciPlex3/GSM4150378_sciPlex3_A549_MCF7_K562_screen_gene.annotations.txt.gz', sep=' ', compression='gzip')
gene_annotations.columns = ['id', 'gene_symbol']
gene_annotations.index += 1
gene_annotations

In [ ]:
p_data = pd.read_csv('sciPlex3/GSM4150378_sciPlex3_pData.txt.gz', sep=' ', compression='gzip').reset_index(drop=True)
p_data.index += 1
p_data

In [ ]:
gene_idx = umi_counts['Gene'].values - 1
cell_idx = umi_counts['Cell'].values - 1
values = umi_counts['UMI'].values

n_genes = gene_annotations.shape[0]
n_cells = p_data.shape[0]

X = csr_matrix((values, (cell_idx, gene_idx)), shape=(n_cells, n_genes))

sciplex3 = anndata.AnnData(X=X)

sciplex3.obs = p_data.set_index('cell')
sciplex3.var = gene_annotations.set_index('id')

sciplex3.write_h5ad('SciPlex3/SciPlex3.h5ad')
sciplex3

In [ ]:
sciplex3 = anndata.read_h5ad('sciPlex3/SciPlex3.h5ad', backed='r')
sciplex3

In [ ]:
gene_lookup = sciplex3.var.reset_index(drop=True)
gene_lookup = gene_lookup['gene_symbol'].to_dict()

cell_lookup = sciplex3.obs.reset_index()['cell'].reset_index().set_index('cell')
cell_lookup = cell_lookup['index'].to_dict()

In [ ]:
cols = ['Combo', 'well_oligo', 'plate_oligo', 'cell_type', 'replicate', 'time_point', 'drug_dose', 'vehicle', ]
cell_meta = sciplex3.obs[
    (sciplex3.obs['top_to_second_best_ratio_W']>=5) & 
    (sciplex3.obs['top_to_second_best_ratio_P']>=5) & 
    (sciplex3.obs['hash_umis_W']>=5) & 
    (sciplex3.obs['hash_umis_P']>=5) & 
    (sciplex3.obs['qval_W']<0.05) & 
    (sciplex3.obs['qval_P']<0.05)
][cols].dropna()
cell_meta

In [ ]:
sciplex3_control_cells = cell_meta[cell_meta['vehicle']]
sciplex3_control_cells

In [ ]:
sciplex3_treated_cells = cell_meta[~cell_meta['vehicle']]
sciplex3_treated_cells.loc[:,'condition'] = sciplex3_treated_cells['cell_type'].astype(str) + ' ' + sciplex3_treated_cells['drug_dose'].astype(str) + ' ' + sciplex3_treated_cells['time_point'].astype(str)
sciplex3_treated_cells

In [ ]:
sciplex3_treated_cells['condition'].nunique()

In [ ]:
sciplex3_treated_cells[['cell_type', 'time_point']].value_counts()

In [ ]:
ax = sns.histplot(sciplex3_treated_cells['condition'].value_counts(), binwidth=8)
ax.set_xlabel('Cells')
ax.set_ylabel('Conditions')

In [ ]:
fig, axs = plt.subplots(3,2, figsize=(15,15))

for i,cell_type in enumerate(sciplex3_treated_cells['cell_type'].unique()):
    for j,time_point in enumerate(sciplex3_treated_cells['time_point'].unique()):
        cells = sciplex3_treated_cells[(sciplex3_treated_cells['cell_type']==cell_type) & (sciplex3_treated_cells['time_point']==time_point)]
        sns.histplot(cells['condition'].value_counts(), binwidth=8, ax=axs[i,j])
        axs[i,j].set_xlim(0,825)
        axs[i,j].set_ylim(0,90)
        axs[i,j].set_title(f'{cell_type}_{time_point}h')
        axs[i,j].set_xlabel('Cells')
        axs[i,j].set_ylabel('Conditions')


In [ ]:
np.random.seed(42)

for condition in tqdm(sciplex3_treated_cells['condition'].unique()):
    condition_cells = sciplex3_treated_cells[sciplex3_treated_cells['condition']==condition]
    condition_control_cells = sciplex3_control_cells[sciplex3_control_cells['cell_type']==condition.split(' ')[0]]

    if condition_cells.shape[0] < 100:
        continue

    condition_index = condition_cells.index.map(cell_lookup).to_numpy()
    control_index = condition_control_cells.index.map(cell_lookup).to_numpy()

    condition_bins = []
    control_bins = []

    for _ in range(10):
        np.random.shuffle(condition_index)
        condition_bins.extend(np.array_split(condition_index, 3))
        control_bins.extend(
            np.random.choice(
                control_index, size=len(condition_index) // 3, replace=False
            )
            for _ in range(3)
        )
    control_frame = pd.DataFrame(np.concatenate([sciplex3.X[control_bins[i]].sum(axis=0).A for i in range(len(control_bins))]).T)
    condition_frame = pd.DataFrame(np.concatenate([sciplex3.X[condition_bins[i]].sum(axis=0).A for i in range(len(condition_bins))]).T)

    control_frame.index = control_frame.index.map(gene_lookup)
    condition_frame.index = condition_frame.index.map(gene_lookup)
    control_frame.columns = [f'{condition} control {i+1}' for i in range(control_frame.shape[1])]
    condition_frame.columns = [f'{condition} {i+1}' for i in range(condition_frame.shape[1])]
    condition_frame = pd.concat([control_frame, condition_frame], axis=1).groupby(level=0).sum()
    
    condition_frame.to_csv(f'sciPlex3Counts/{condition}.tsv.gz', sep='\t', compression='gzip')

In [ ]:
gene_info = pd.read_csv('https://ftp.ncbi.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz', compression='gzip', sep='\t')
gene_info = gene_info[(gene_info['#tax_id']==9606) & (gene_info['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

In [ ]:
def cd_signature(control, treatment, data, metadata, meta_class_column_name):
    
    ctrl_ids = metadata.loc[metadata[meta_class_column_name]==control, :].index.tolist() #control
    case_ids = metadata.loc[metadata[meta_class_column_name]==treatment,:].index.tolist() #case
    
    signature = characteristic_direction(
        data.loc[:, ctrl_ids], 
        data.loc[:, case_ids], 
        calculate_sig=True
    )
    signature = signature.sort_values("CD-coefficient", ascending=False)
    signature['Significance'] = signature['CD-coefficient'].apply(abs)
    
    return signature

In [ ]:
signatures = {}

for matrix in tqdm(os.listdir('sciPlex3Counts/')):
    if 'tsv.gz' not in matrix:
        continue
    condition = matrix.replace('.tsv.gz','')

    meta_class_column_name = 'Class'
    control_name = 'Control'
    treatment_name = condition

    # Load expression DF
    expr_df = pd.read_csv(f"sciPlex3Counts/{matrix}", sep="\t", compression="gzip", index_col=0).sort_index()

    # Create metadata DF
    meta_df = expr_df.T.drop(columns=expr_df.index)
    meta_df['Sample'] = meta_df.index
    meta_df['Class'] = meta_df['Sample'].map(lambda x: 'Control' if 'control' in x else condition)
    

    # Map gene symbols
    expr_df.index = expr_df.index.map(str.upper).map(synonyms)
    expr_df = expr_df.reset_index().dropna().set_index('index').groupby(level=0).sum()
    ## Filter out non-expressed genes
    expr_df = expr_df.loc[expr_df.sum(axis=1) > 0, :]
    ## Filter out lowly expressed genes
    #expr_df = filter_by_expr(expr_df, group=meta_df[meta_class_column_name].apply(lambda x: x==treatment_name))

    # Run characteristic direction
    signatures[condition] = cd_signature(control_name, treatment_name, expr_df, meta_df, meta_class_column_name)
    signatures[condition].to_csv(f'sciPlex3CD/{condition}.tsv.gz', sep='\t', compression='gzip')


In [ ]:
cd_signatures = pd.concat([signature['CD-coefficient'] for signature in signatures.values()], axis=1)
cd_signatures.columns = signatures.keys()
cd_signatures

In [ ]:
sns.histplot(cd_signatures.stack(), binwidth=0.001)

In [ ]:
def rename_perturbation(pert):
    (cell_type, sample_dose, timepoint) = pert.split(' ')
    drug = sample_metadata.loc[sample_dose, 'name'].split(' (')[0].strip().replace('(+)-','')
    dose = int(sample_metadata.loc[sample_dose, 'dose'])
    if dose >= 1000:
        dose //= 1000
        dose = f'{dose}µM'
    else:
        dose = f'{dose}nM'

    return f'{cell_type}_{drug}_{dose}_{int(float(timepoint))}h'

In [ ]:
sciplex3_signatures = pd.concat([
    cd_signatures.apply(lambda x: x[x>0].nlargest(500), axis=0).stack(), 
    cd_signatures.apply(lambda x: x[x<0].nsmallest(500), axis=0).stack()
]).reset_index()
sciplex3_signatures.columns = ['Gene', 'Perturbation', 'CD']
sciplex3_signatures['Perturbation'] = sciplex3_signatures['Perturbation'].map(rename_perturbation)
sciplex3_signatures['Threshold'] = sciplex3_signatures['CD'].map(lambda x: 1 if x>0 else -1)
sciplex3_signatures

## Prepare Data for SQL Ingestion

### Resource

In [ ]:
# (id, name, long_description, short_description, url, num_attributes, num_datasets)
(123, 'Sci-Plex', 'Global quantitative transcriptional response to independent perturbations at single cell resolution, using nuclear hashing to enable single-cell combinatorial indexing.', 'sciRNA-seq profiling of perturbation response across multiple cancer cell lines.', 'https://github.com/cole-trapnell-lab/sci-plex', 2302, 1)

### Dataset

In [ ]:
# id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, positive_association, negative_association, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural, is_archived
(194, 'Sci-Plex Drug Perturbation Signatures', 'Drug Perturbation Signatures', 'sciRNA-seq expression profiles for A549, K562, and MCF7 cells treated with 188 compounds at 4 doses', 'gene-drug perturbation associations by differential expression of gene following drug perturbation', 'genes differentially expressed following the {0} drug perturbation from the Sci-Plex Drug Perturbation Signatures dataset.', 'sets of genes differentially expressed following drug perturbation from the Sci-Plex Drug Perturbation Signatures dataset.', 'drug perturbations changing expression of {0} gene from the Sci-Plex Drug Perturbation Signatures dataset.', 'increased expression', 'decreased expression', 1, 1, '2025-11-11', 'sciplex', 0, 123, 16, 7, 8, 2, 'gene expression by RNA-seq', 'primary experimental data', 'high throughput, data driven', 'drug perturbations')

### Publication

In [ ]:
#(id, long_citation, short_citation, url, pmid, pubmed_url, first_author_last_name, first_author_initials, journal_abbreviation, year, title, volume, pages)
(176, 'Srivatsan SR, et al. (2019) Massively multiplex chemical transcriptomics at single-cell resolution. Science. 367(6473):45-51', 'Srivatsan, Science, 2019', 'dx.doi.org/10.1126/science.aax6234', 31806696, 'https://pubmed.ncbi.nlm.nih.gov/31806696', 'Srivatsan', 'SR', 'Science', 2019, 'Massively multiplex chemical transcriptomics at single-cell resolution', 367, '45-51')

(283, 194, 176)

### Naming Authority

In [ ]:
# (id, name, description, url, publication_fk)
(118, 'Sci-Plex', '', 'sciRNA-seq profiling of perturbation response across multiple cancer cell lines.', 'https://github.com/cole-trapnell-lab/sci-plex', 176)

### Attribute

In [ ]:
attributes = pd.read_csv('../../tables/attribute.txt', sep='\t')
attributes['name_from_naming_authority'] = attributes['name_from_naming_authority'].str.lower().str.strip()
attributes = attributes.set_index('name_from_naming_authority')
attributefks = attributes['id'].to_dict()
index = 562952

for pert in sciplex3_signatures['Perturbation'].unique():
    if pert.lower() not in attributes.index:
        cell_line, drug, dose, timepoint = pert.split('_')
        print((index, pert, f'{drug} {dose} perturbation in {cell_line} cell line at {timepoint} timepoint', 118), end=',\n')
        attributefks[pert.lower()] = index
        index += 1

### Gene Set

In [ ]:
index = 139500000
genesetfks = {}

for pert in sciplex3_signatures['Perturbation'].unique():
    cell_line, drug, dose, timepoint = pert.split('_')
    print((index, pert, f'{drug} {dose} perturbation in {cell_line} cell line at {timepoint} timepoint', 194, 8, attributefks[pert.lower()]), end=',\n')
    genesetfks[pert] = index
    index += 1

### Associations

In [ ]:
index = 99000000

genes = pd.read_csv('../../tables/gene.txt', sep='\t')
genes['symbol'] = genes['symbol'].str.upper()
genes = genes.set_index('symbol')
genefks = genes['id'].to_dict()

associations = sciplex3_signatures.copy()
associations['Gene'] = associations['Gene'].apply(lambda x: genefks[x])
associations['Perturbation'] = associations['Perturbation'].apply(lambda x: genesetfks[x])
associations = associations.get(['Gene', 'Perturbation', 'CD', 'Threshold'])
associations.columns = ['gene_fk', 'gene_set_fk', 'standardized_value', 'threshold_value']
associations = associations.drop_duplicates().reset_index(drop=True)
associations.index += index
associations = associations.rename_axis('id')
associations.to_csv('../../harmonizome-update/sciplex.csv')
associations

In [ ]:
associations.nunique()

## Create Downloads

In [ ]:
output_path = 'downloads'

### Binary Matrix

In [ ]:
binarymatrix = pd.crosstab(sciplex3_signatures['Gene'], sciplex3_signatures['Perturbation'], values=sciplex3_signatures['Threshold'], aggfunc=np.mean).replace(np.nan, 0).astype(int)
binarymatrixT = binarymatrix.T
binarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
binarymatrix

### Gene Attribute Edge List

In [ ]:
edgelist = sciplex3_signatures.copy()
edgelist['Gene ID'] = edgelist['Gene'].map(geneids).astype(int)
edgelist = edgelist[['Gene', 'Gene ID', 'Perturbation', 'CD', 'Threshold']].reset_index(drop=True)
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['Perturbation']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Up Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_up_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Down Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_dn_crisp.gmt', 'w') as f:
    arr = binarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = binarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrix.index[arr[:, i] == -1]])>= 5:
            print(attributes[i], '', *binarymatrix.index[arr[:, i] == -1], sep='\t', end='\n', file=f)

### Up Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_up_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Down Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_dn_crisp.gmt', 'w') as f:
    arr = binarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = binarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*binarymatrixT.index[arr[:, i] == -1]])>= 5:
            print(genes[i], '', *binarymatrixT.index[arr[:, i] == -1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(binarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=binarymatrix.index, columns=binarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(binarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=binarymatrixT.index, columns=binarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Standardized Matrix

In [ ]:
standardizedmatrix = cd_signatures.rename(lambda x: rename_perturbation(x), axis=1).rename_axis('Gene').rename_axis('Perturbation', axis=1)
standardizedmatrix.to_csv(f'{output_path}/gene_attribute_matrix_standardized.txt.gz', sep='\t', compression='gzip')
standardizedmatrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for attr in attributeslist.index:
    attr = attributeslist.loc[attr]
    nodes[f"Sci-Plex_{attr['Perturbation']}"] = {
        "type":"drug perturbation",
        "properties": {
            "label":attr['Perturbation'],
            "id":f"Sci-Plex_{attr['Perturbation']}"
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    edges.append({
        "source": f"Sci-Plex_{attr['Perturbation']}",
        "relation": "increases expression of" if edge['Threshold'] == 1 else 'represses expression of',
        "target": int(edge['Gene ID']),
        "properties":{
            "id": f"Sci-Plex_{attr['Perturbation']}"+":"+str(edge['Gene ID']),
            "source_label":edge['Perturbation'],
            "source_id": f"Sci-Plex_{attr['Perturbation']}",
            "target_id": int(edge['Gene ID']),
            "target_label":edge['Gene'],
            "directed":True,
            "standardized":edge['CD'],
            "threshold": int(edge['Threshold']),
        }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/sciplex.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    
    print('', file=f)
    for edge in edges:
        print(f"<{edge['properties']['source_id']}>", f"{'RO0003003' if edge['properties']['threshold']==1 else 'RO:0003002'}", f"gene:{edge['properties']['target_id']}", end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/sciplex.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
namespace = {'gene':'NCBI Entrez', 'drug perturbation':'Sci-Plex'}
nodeframe['namespace'] = nodeframe['type'].map(namespace)
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/sciplex_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['standardized'] = edgeframe['properties'].apply(lambda x: x['standardized'])
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe = edgeframe.get(['source', 'relation', 'target','standardized', 'threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/sciplex_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
binarymatrix = pd.read_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip', index_col=0)

In [ ]:
ax = sns.clustermap(binarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
ax = sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file, dir):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[term+'_'+dir] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_up_crisp.gmt', 'r'), 'up')
libdict = libdict | load_gmt(open(f'{output_path}/gene_set_library_dn_crisp.gmt', 'r'), 'down')
scatterdir = 'images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the Sci-Plex Drug Perturbation Signatures Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 4, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=5,mindist=0.1
     ,spread=3.5
     ,maxdf=0.5
     ,mindf=1
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in the Sci-Plex Drug Perturbation Signatures Library')
save(plot)